In [ ]:
from pathlib import Path
import random
import shutil
import yaml

random.seed(42)

# CONFIGURAÇÃO PARA O DATASET DE CUBOS (classificação)
dataset_path = Path(r"C:\Users\User\Documents\IA_PROJECT\Archives\files_webot\IA_20252\controllers\dataset_generator\dataset_cubes_only")
output_path = Path(r"C:\Users\User\Documents\IA_PROJECT\Archives\files_webot\IA_20252\controllers\Model_Processing\cubes_classification_folds")
k = 5  # número de folds

# ✅ CONFIGURAÇÃO DE DIVISÃO
TRAIN_RATIO = 0.70  # 70% para treino
VAL_RATIO = 0.15    # 15% para validação  
TEST_RATIO = 0.15   # 15% para teste

print(f"[INFO] Dataset de cubos: {dataset_path}")
print(f"[INFO] Output folds: {output_path}")
print(f"[INFO] Divisão: {TRAIN_RATIO*100}% Train, {VAL_RATIO*100}% Val, {TEST_RATIO*100}% Test")

# LER LABELS.TXT E CRIAR PARES (imagem, label)
labels_file = dataset_path / "labels.txt"
if not labels_file.exists():
    print(f"[ERROR] Arquivo {labels_file} não encontrado!")
    exit(1)

pairs = []
label_counts = {"red": 0, "green": 0, "blue": 0}

with open(labels_file, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        
        # Formato: img_000001.png red
        parts = line.split()
        if len(parts) >= 2:
            img_name = parts[0]
            label_name = parts[1]
            
            img_path = dataset_path / img_name
            if img_path.exists() and label_name in ["red", "green", "blue"]:
                pairs.append((img_path, label_name))
                label_counts[label_name] += 1

print(f"[INFO] Dataset original:")
for color, count in label_counts.items():
    print(f"  {color}: {count} imagens")

# ✅ BALANCEAR O DATASET - MESMO NÚMERO DE IMAGENS POR CLASSE
print(f"\n[INFO] Balanceando dataset...")

# Separar por classe
red_pairs = [(img, label) for img, label in pairs if label == "red"]
green_pairs = [(img, label) for img, label in pairs if label == "green"] 
blue_pairs = [(img, label) for img, label in pairs if label == "blue"]

# ✅ PEGAR O MENOR NÚMERO COMO REFERÊNCIA
min_samples = min(len(red_pairs), len(green_pairs), len(blue_pairs))
print(f"Usando {min_samples} imagens por classe (menor quantidade)")

# ✅ EMBARALHAR E PEGAR APENAS min_samples DE CADA CLASSE
random.shuffle(red_pairs)
random.shuffle(green_pairs)
random.shuffle(blue_pairs)

balanced_pairs = (
    red_pairs[:min_samples] + 
    green_pairs[:min_samples] + 
    blue_pairs[:min_samples]
)

# ✅ EMBARALHAR O DATASET BALANCEADO
random.shuffle(balanced_pairs)

# ✅ ATUALIZAR VARIÁVEIS
pairs = balanced_pairs
n = len(pairs)
label_counts = {"red": min_samples, "green": min_samples, "blue": min_samples}

print(f"\nDataset balanceado:")
for color, count in label_counts.items():
    print(f"  {color}: {count} imagens")
print(f"Total: {n} imagens (perfeitamente balanceado, tal como tudo deveria ser...)")

if n == 0:
    print("[ERROR] Nenhum par válido encontrado!")
    exit(1)

test_size = int(n * TEST_RATIO)
remaining_size = n - test_size

# ✅ DIVIDIR RESTANTE ENTRE TRAIN/VAL PARA CADA FOLD
# Train: 70% do total, Val: 15% do total, TEST: 15%
train_val_ratio = TRAIN_RATIO / (TRAIN_RATIO + VAL_RATIO)  # 70/85 = 0.824
val_train_ratio = VAL_RATIO / (TRAIN_RATIO + VAL_RATIO)   # 15/85 = 0.176

print(f"\n[INFO] Calculando divisões:")
print(f"  Teste global: {test_size} imagens ({TEST_RATIO*100}%)")
print(f"  Para K-Fold: {remaining_size} imagens")
print(f"  Por fold - Train: {train_val_ratio*100:.1f}%, Val: {val_train_ratio*100:.1f}%")

test_pairs = []
train_val_pairs = []

# Separar teste por classe para manter balanceamento
for color in ["red", "green", "blue"]:
    color_pairs = [(img, label) for img, label in pairs if label == color]
    color_test_size = test_size // 3  # 1/3 do teste para cada classe
    
    test_pairs.extend(color_pairs[:color_test_size])
    train_val_pairs.extend(color_pairs[color_test_size:])

print(f"\n[INFO] Divisão final:")
print(f"  Teste: {len(test_pairs)} imagens")
print(f"  Train+Val: {len(train_val_pairs)} imagens")


# Separar train_val_pairs por classe
red_train_val = [(img, label) for img, label in train_val_pairs if label == "red"]
green_train_val = [(img, label) for img, label in train_val_pairs if label == "green"] 
blue_train_val = [(img, label) for img, label in train_val_pairs if label == "blue"]

# Embaralhar cada classe separadamente
random.shuffle(red_train_val)
random.shuffle(green_train_val)
random.shuffle(blue_train_val)

# Criar folds distribuindo cada classe igualmente
folds = []
for i in range(k):
    # Distribuir cada classe usando modulo para garantir distribuição uniforme
    fold_red = red_train_val[i::k]    # Pega 1 a cada k elementos começando do índice i
    fold_green = green_train_val[i::k]
    fold_blue = blue_train_val[i::k]
    
    # Combinar as classes e embaralhar o fold
    fold_data = fold_red + fold_green + fold_blue
    random.shuffle(fold_data)
    folds.append(fold_data)

print(f"[INFO] Verificando balanceamento dos folds:")
for i, fold in enumerate(folds):
    fold_counts = {"red": 0, "green": 0, "blue": 0}
    for _, label in fold:
        fold_counts[label] += 1
    print(f"  Fold {i}: {len(fold)} imagens - red:{fold_counts['red']}, green:{fold_counts['green']}, blue:{fold_counts['blue']}")
print(f"\n[INFO] Criando {k} folds para classificação:")
for i, fold in enumerate(folds):
    fold_counts = {"red": 0, "green": 0, "blue": 0}
    for _, label in fold:
        fold_counts[label] += 1
    print(f"  Fold {i}: {len(fold)} imagens - red:{fold_counts['red']}, green:{fold_counts['green']}, blue:{fold_counts['blue']}")

# CRIAR ESTRUTURA COM TRAIN/VAL/TEST PARA CADA FOLD
for i in range(k):
    fold_name = f"fold_{i}"
    fold_root = output_path / fold_name

    print(f"\n[INFO] Criando fold {i} em {fold_root}")

    # CRIAR DIRETÓRIOS POR CLASSE (train/val/test + red/green/blue)
    for split in ["train", "val", "test"]:
        for class_name in ["red", "green", "blue"]:
            (fold_root / split / class_name).mkdir(parents=True, exist_ok=True)

    val_pairs = folds[i]
    train_pairs = []
    for j, other_fold in enumerate(folds):
        if j != i:
            train_pairs.extend(other_fold)

    print(f"  Train: {len(train_pairs)} imagens")
    print(f"  Val: {len(val_pairs)} imagens")
    print(f"  Test: {len(test_pairs)} imagens")

    for split, split_pairs in [("train", train_pairs), ("val", val_pairs), ("test", test_pairs)]:
        split_counts = {"red": 0, "green": 0, "blue": 0}
        
        for img_path, label_name in split_pairs:
            # COPIAR PARA PASTA DA CLASSE: fold_X/train/red/img_000001.png
            dst_img = fold_root / split / label_name / img_path.name
            shutil.copy2(img_path, dst_img)
            split_counts[label_name] += 1
        
        print(f"    {split.capitalize()}: red={split_counts['red']}, green={split_counts['green']}, blue={split_counts['blue']}")

    # CRIAR LABELS.TXT PARA CADA SPLIT
    for split, split_pairs in [("train", train_pairs), ("val", val_pairs), ("test", test_pairs)]:
        labels_txt_path = fold_root / f"{split}_labels.txt"
        with open(labels_txt_path, "w", encoding="utf-8") as f:
            for img_path, label_name in split_pairs:
                rel_path = f"{split}/{label_name}/{img_path.name}"
                f.write(f"{rel_path} {label_name}\n")

    data_yaml = {
        "fold_id": i,
        "total_folds": k,
        "dataset_type": "image_classification",
        "classes": ["red", "green", "blue"],
        "num_classes": 3,
        "train_samples": len(train_pairs),
        "val_samples": len(val_pairs),
        "test_samples": len(test_pairs),
        "split_ratios": {
            "train": f"{len(train_pairs)/n*100:.1f}%",
            "val": f"{len(val_pairs)/n*100:.1f}%", 
            "test": f"{len(test_pairs)/n*100:.1f}%"
        },
        "folder_structure": {
            "train": "train/[class]/[images]",
            "val": "val/[class]/[images]",
            "test": "test/[class]/[images]"
        },
        "labels_files": {
            "train": "train_labels.txt",
            "val": "val_labels.txt",
            "test": "test_labels.txt"
        }
    }

    yaml_path = fold_root / f"fold_{i}_info.yaml"
    with open(yaml_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(data_yaml, f, sort_keys=False)

    print(f"  ✓ YAML criado: {yaml_path}")
    print(f"  ✓ Labels: train_labels.txt, val_labels.txt, test_labels.txt")

print(f"\n[ESTRUTURA] Cada fold tem:")
print(f"  fold_X/")
print(f"  ├── train/")
print(f"  │   ├── red/")
print(f"  │   ├── green/")
print(f"  │   └── blue/")
print(f"  ├── val/")
print(f"  │   ├── red/")
print(f"  │   ├── green/")
print(f"  │   └── blue/")
print(f"  ├── test/")
print(f"  │   ├── red/")
print(f"  │   ├── green/")
print(f"  │   └── blue/")
print(f"  ├── train_labels.txt")
print(f"  ├── val_labels.txt")
print(f"  ├── test_labels.txt")
print(f"  └── fold_X_info.yaml")


[INFO] Dataset de cubos: C:\Users\User\Documents\IA_PROJECT\Archives\files_webot\IA_20252\controllers\dataset_generator\dataset_cubes_only
[INFO] Output folds: C:\Users\User\Documents\IA_PROJECT\Archives\files_webot\IA_20252\controllers\Model_Processing\cubes_classification_folds
[INFO] Divisão: 70.0% Train, 15.0% Val, 15.0% Test
[INFO] Dataset original:
  red: 9216 imagens
  green: 4607 imagens
  blue: 3456 imagens

[INFO] Balanceando dataset...
Usando 3456 imagens por classe (menor quantidade)

Dataset balanceado:
  red: 3456 imagens
  green: 3456 imagens
  blue: 3456 imagens
Total: 10368 imagens (perfeitamente balanceado, tal como tudo deveria ser...)

[INFO] Calculando divisões:
  Teste global: 1555 imagens (15.0%)
  Para K-Fold: 8813 imagens
  Por fold - Train: 82.4%, Val: 17.6%

[INFO] Divisão final:
  Teste: 1554 imagens
  Train+Val: 8814 imagens
[INFO] Verificando balanceamento dos folds:
  Fold 0: 1764 imagens - red:588, green:588, blue:588
  Fold 1: 1764 imagens - red:588, gr

: 